# Autograd - Build
This is the build section of the project, where new functions, classes, etc are created and tested. Finished pieces are moved into main where they are then called and used by build.

## Setup

In [ ]:
import numpy as np
import main
from typing import Union, List
from matplotlib import pyplot as plt
import matplotlib_inline
import pandas as pd
%matplotlib

## Build

## Training zone

### Importing training data

In [ ]:
# Organising the data
data = pd.read_csv(r"C:\Users\acart\OneDrive\Desktop\CodingProjects\mnist\mnist_train.csv")

data = np.array(data)
m, n = data.shape
np.random.shuffle(data)

data_dev = data[0:10000]
Y_dev = data_dev[:, 0]
X_dev = data_dev[:, 1:] / 255.0

data_train = data[10000:m]
Y_train = data_train[:, 0]
X_train = data_train[:, 1:] / 255.0

print(X_train.shape)
print(Y_train.shape)

In [ ]:
# one hot gives: (num_samples, 10)
def one_hot(Y):
    Y = np.asarray(Y, dtype=int)
    one_hot_Y = np.zeros((Y.size, Y.max() + 1), dtype=np.float32)
    one_hot_Y[np.arange(Y.size), Y] = 1
    return one_hot_Y

Y_train = one_hot(Y_train)
Y_dev = one_hot(Y_dev)

In [ ]:
print(X_train.shape)   # should be (num_samples, 784)
print(Y_train.shape)   # should be (num_samples, 10)

### Creating a Model

In [ ]:
model = main.Model(input_size=784, output_size=10, hidden_size=128,
              number_of_layers=4, activation_function=main.ReLU, normalisation_function=main.Softmax,
              random_seed=0, initialisation_function="He")

### Training a model

In [ ]:
# **Hyperparameters **
num_epochs = 100
batch_size = 64
learning_rate = 1e-3

custom_lr_scheduler = [[1, 1e-3], [20, 5e-4]]

In [ ]:
training_loss_over_time = []
validation_loss_over_time = []
validation_accuracy_over_time = []
epoch = 0

In [ ]:
# Getting a recording of the model before any training
train_output = model.forward(X_train)
train_loss = main.CrossEntropyLoss(train_output, Y_train)
training_loss_over_time.append(train_loss)

train_predictions = np.argmax(train_output, axis=1)
train_targets = np.argmax(Y_train, axis=1)
train_accuracy = np.mean(train_predictions == train_targets) * 100


val_output = model.forward(X_dev)
val_loss = main.CrossEntropyLoss(val_output, Y_dev)
validation_loss_over_time.append(val_loss)

val_predictions = np.argmax(val_output, axis=1)
val_targets = np.argmax(Y_dev, axis=1)
val_accuracy = np.mean(val_predictions == val_targets) * 100
validation_accuracy_over_time.append(val_accuracy)

print(
    f"Initial model stats: Train Loss: {round(train_loss, 2):5} | Train Acc: {round(train_accuracy, 3):6}% | "
    f"Val Loss: {round(val_loss, 3):5} | Val Acc: {round(val_accuracy, 3):6}%"
    ) 

In [ ]:
# Main training loop
for i in range(num_epochs):
    epoch += 1
    epoch_loss = 0

    # Choosing learning rate based on custom scheduler
    if custom_lr_scheduler is not None:
        if custom_lr_scheduler and custom_lr_scheduler[0][0] == epoch:
            learning_rate = custom_lr_scheduler[0][1]
            custom_lr_scheduler.pop(0)

    # Shuffling the training data
    random_order = np.random.permutation(len(X_train))
    X_train = X_train[random_order]
    Y_train = Y_train[random_order]

    num_batches = int(np.ceil(len(X_train) / batch_size))

    # Training the model on the batched dataset
    for b in range(num_batches):
        start = b * batch_size
        end = min(start + batch_size, len(X_train))

        batch_output = model.forward(X_train[start:end])
        epoch_loss += main.CrossEntropyLoss(batch_output, Y_train[start:end])

        model.backwards(batch_output, Y_train[start:end], main.CrossEntropyLoss)
        model.update_parameters(learning_rate)


    # Passing the full datasets into the model to record accurate loss and accuracy
    train_output = model.forward(X_train)
    train_loss = main.CrossEntropyLoss(train_output, Y_train)
    training_loss_over_time.append(train_loss)

    train_predictions = np.argmax(train_output, axis=1)
    train_targets = np.argmax(Y_train, axis=1)
    train_accuracy = np.mean(train_predictions == train_targets) * 100


    val_output = model.forward(X_dev)
    val_loss = main.CrossEntropyLoss(val_output, Y_dev)
    validation_loss_over_time.append(val_loss)

    val_predictions = np.argmax(val_output, axis=1)
    val_targets = np.argmax(Y_dev, axis=1)
    val_accuracy = np.mean(val_predictions == val_targets) * 100
    validation_accuracy_over_time.append(val_accuracy)

    # Printing the results of the current epoch
    print(
        f"Epoch: {epoch:3} | Train Loss: {round(train_loss, 2):5} | Train Acc: {round(train_accuracy, 3):6}% | "
        f"Val Loss: {round(val_loss, 3):5} | Val Acc: {round(val_accuracy, 3):6}% | Lr: {learning_rate:5}"
        )   

### Displaying results

In [ ]:
# matplotlib plot
x = np.linspace(0, len(training_loss_over_time), len(training_loss_over_time))
y1 = training_loss_over_time
y2 = validation_loss_over_time
y3 = validation_accuracy_over_time

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].set_title("Loss over time")
axes[0].plot(x, y1, label="Training loss", linestyle="-", color="red")
axes[0].plot(x, y2, label="Validation loss", linestyle="-", color="black")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_ylim(bottom=0)
axes[0].grid(True)
axes[0].legend()

axes[1].set_title("Validation accuracy over time")
axes[1].plot(x, y3, label="Validation accuracy", linestyle="--", color="orange")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)")
axes[1].set_ylim(0, 100)
axes[1].grid(True)
axes[1].legend()

plt.tight_layout()
plt.show()

### Saving a model
This code is left hashed out to avoid accidentaly overwriting the saved model file

In [ ]:
import pickle
import pathlib

parameters_path = pathlib.Path("pretrained_model_parameters.pkl")

if input("Are you sure you want to overwrite the existing saved model? (Y/N)").lower == "y" or "yes":
    with parameters_path.open("wb") as file:
        pickle.dump(model.parameters, file)

    print("Overwrote file, saved model parameters")